![CREWS UFFFS Nowcast](logo.png)

# Tutorial 3: Interpret, validate, and apply your GSMaP nowcast

This notebook continues from `2_Nowcast_GSMaP_Rainfall.ipynb`. It assumes you have already produced and saved an ensemble nowcast NetCDF in `data/gsmap/nowcast/`, and still have the processed GSMaP NetCDF from Tutorial 1 in `data/gsmap/processed/`.

Here you will:

- Compute **ensemble statistics** and **exceedance probabilities**.
- Zoom in around **Vientiane** and **Phnom Penh**.
- Create an animation of the nowcast.
- Extract **point time series** and **threshold-probability tables** for each city.
- Compare GSMaP and the nowcast against a **rain-gauge CSV**.
- Optionally **validate** the nowcast against withheld GSMaP observations.
- Export operational products to **GeoTIFF**.

# Setup

The cells below reproduce the project settings, folder structure, and helper functions from Tutorial 2, then load the saved GSMaP processed observations and the saved nowcast from disk. This makes Tutorial 3 self-contained — you can re-run it independently from Tutorial 2.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import imageio.v2 as imageio
import rioxarray  # noqa: F401. Required for GeoTIFF export with xarray/rioxarray.

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Libraries imported.")


## Step 2a: Select the event time period

This is the cell most workshop participants will edit during exercises.

Define the **start** and **end** time of the event you want to analyse.
Use the same window as the one used in Tutorials 1 and 2 if you want
the interpretation step to use matching observations and forecasts.

- All times are in **UTC**.
- GSMaP is hourly, so use full-hour timestamps (`HH:00:00`).
- The end time is **exclusive**.


In [ ]:
# ---------------------------------------------------------------------
# Edit these two lines to change the event period.
# ---------------------------------------------------------------------
EVENT_START = "2025-09-16T00:00:00Z"   # inclusive, UTC
EVENT_END   = "2025-09-16T12:00:00Z"   # exclusive, UTC

# ---------------------------------------------------------------------
# Quick sanity check on the selected period.
# ---------------------------------------------------------------------
_start_dt = pd.to_datetime(EVENT_START, utc=True)
_end_dt   = pd.to_datetime(EVENT_END,   utc=True)
_n_hours  = int((_end_dt - _start_dt).total_seconds() // 3600)

if _end_dt <= _start_dt:
    raise ValueError(
        f"EVENT_END ({EVENT_END}) must be strictly after EVENT_START ({EVENT_START})."
    )

print(f"Event start (UTC):     {_start_dt}")
print(f"Event end   (UTC):     {_end_dt}")
print(f"Duration:              {_n_hours} hours")
print(f"Expected GSMaP fields: {_n_hours} hourly images")


## Step 2b: Define the remaining settings

The cell below holds all other settings: bounding box, plotting options,
nowcast-related parameters, and city zoom locations.

The **time period** is taken from `EVENT_START` and `EVENT_END` in the
cell above, so participants only need to touch one place to change the
event window.


In [ ]:
settings = {
    # ---------------------------------------------------------------------
    # General workshop metadata
    # ---------------------------------------------------------------------
    "project_name": "CREWS_Lao_PDR_training",
    "country_or_region": "Southeast Asia example domain",
    "event_name": "Example heavy rainfall event",

    # ---------------------------------------------------------------------
    # Google Earth Engine settings
    # ---------------------------------------------------------------------
    "gee_project": None,
    "ee_collection": "JAXA/GPM_L3/GSMaP/v7/operational",
    "band": "hourlyPrecipRate",

    # ---------------------------------------------------------------------
    # Spatial settings
    # ---------------------------------------------------------------------
    "bbox": [95.0, -12.0, 155.0, 20.0],

    # City-focused zoom plots.
    "city_zoom_enabled": True,
    "city_zoom_buffer_degrees": 2.0,
    "city_zoom_locations": [
        {
            "name": "Phnom Penh",
            "country": "Cambodia",
            "lat": 11.5564,
            "lon": 104.9282,
            "notes": "City-centre reference point; adjust if your project uses a specific station or district.",
        },
        {
            "name": "Vientiane",
            "country": "Lao PDR",
            "lat": 17.9757,
            "lon": 102.6331,
            "notes": "City-centre reference point; adjust if your project uses a specific station or district.",
        },
    ],

    # ---------------------------------------------------------------------
    # Plotting options
    # ---------------------------------------------------------------------
    "plot_zero_as_nan": True,
    "plot_use_osm_basemap": True,
    "plot_basemap_alpha": 0.65,

    # Transparency of the rainfall layer (0 = fully transparent, 1 = fully opaque).
    # A value below 1 lets the basemap (coastlines, rivers, borders, cities)
    # remain visible underneath the rainfall colours.
    "plot_rainfall_alpha": 0.7,

    # Colour-scale upper bound (vmax) handling.
    # Rainfall has a long-tailed distribution: a single extreme pixel can
    # dominate auto-scaling and wash out the rest of the field. We therefore
    # compute vmax from a high percentile of the positive values, and use
    # the SAME vmax across plots of the same kind so they can be compared.
    "plot_vmax_percentile": 99.0,
    "plot_intensity_vmax":  None,   # mm/h, manual override; None = auto from percentile
    "plot_spread_vmax":     None,   # mm/h, manual override for ensemble spread plots

    # ---------------------------------------------------------------------
    # Event time period (defined in the cell above)
    # ---------------------------------------------------------------------
    "start_time": EVENT_START,
    "end_time":   EVENT_END,

    "nowcast_reference_time": None,
    "download_scale_m": 11132,

    # ---------------------------------------------------------------------
    # Nowcast settings
    # ---------------------------------------------------------------------
    "n_input_files": 6,
    "n_lead_times": 6,
    "timestep_minutes": 60,
    "rain_threshold_mm_h": 0.1,
    "use_steps_if_available": True,
    "n_ensemble_members": 8,
    "km_per_pixel": 10.0,
    "random_seed": 42,
}

settings


In [ ]:
# Create folders for the processed input data, nowcast output, figures, and tables.
data_folder = Path("./gsmap_data")
processed_folder = data_folder / "processed_netcdf"
nowcast_folder = data_folder / "nowcast"
figures_folder = data_folder / "figures"

for folder in [processed_folder, nowcast_folder, figures_folder]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Processed NetCDF folder: {processed_folder.resolve()}")
print(f"Nowcast output folder: {nowcast_folder.resolve()}")
print(f"Figures folder: {figures_folder.resolve()}")

In [ ]:
def prepare_field_for_plot(da, zero_as_nan=None):
    """Return a copy of a DataArray that is convenient for plotting.

    For rainfall maps, values of exactly 0 often dominate the figure even though
    they simply mean 'no rain'. During a workshop, it is usually clearer to make
    these dry pixels transparent and let the basemap show through.

    This function only affects the plotted field. It does not modify the original
    xarray object or the values saved in NetCDF/GeoTIFF outputs.
    """
    if zero_as_nan is None:
        zero_as_nan = settings.get("plot_zero_as_nan", True)

    field = da.squeeze()

    if zero_as_nan:
        field = field.where(field != 0)

    return field


def get_transparent_colormap(cmap_name="Blues"):
    """Return a matplotlib colormap where NaN values are transparent."""
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad((1, 1, 1, 0))
    return cmap


def compute_vmax(data_arrays, percentile=None, override=None):
    """Compute a robust shared vmax across one or more DataArrays.

    Rainfall fields have very long-tailed distributions: one extreme pixel can
    dominate the colour bar and wash out the rest of the field. We therefore
    use a high percentile of all positive, finite values across the inputs,
    rather than the absolute maximum.

    Passing several fields here is what creates a SHARED colour scale, so the
    resulting plots can be compared visually.
    """
    if override is not None:
        return float(override)
    if percentile is None:
        percentile = settings.get("plot_vmax_percentile", 99.0)

    arrays = []
    for da in data_arrays:
        arr = np.asarray(da.values).ravel()
        arr = arr[np.isfinite(arr) & (arr > 0)]
        if arr.size:
            arrays.append(arr)

    if not arrays:
        return None

    combined = np.concatenate(arrays)
    return float(np.percentile(combined, percentile))


def add_openstreetmap_basemap(ax, enabled=None, alpha=None):
    """Add an OpenStreetMap basemap to a lon/lat matplotlib axis if possible.

    The default ``contextily.add_basemap(ax, crs="EPSG:4326", ...)`` call routes
    the tile reprojection through rasterio/GDAL. On some installations
    (especially Windows with mixed PROJ/GDAL versions), this fails with
    ``"The WKT could not be parsed. OGR Error code 6"``.

    To avoid that, we use ``contextily.bounds2img(..., ll=True)`` to download
    the OSM tiles using lon/lat bounds, then reproject the returned Web
    Mercator extent to lon/lat with pyproj before placing the image with
    ``ax.imshow``. No GDAL WKT parsing is involved.

    The slight horizontal/vertical distortion from displaying Web Mercator
    tiles on a lon/lat axis is negligible at tropical and mid-latitudes.
    """
    if enabled is None:
        enabled = settings.get("plot_use_osm_basemap", True)
    if alpha is None:
        alpha = settings.get("plot_basemap_alpha", 0.65)

    if not enabled:
        return

    try:
        import contextily as ctx
    except ImportError:
        print(
            "OpenStreetMap basemap skipped: contextily is not installed. "
            "Install it with: pixi add contextily xyzservices"
        )
        return

    try:
        from pyproj import Transformer
    except ImportError:
        print("OpenStreetMap basemap skipped: pyproj is not available.")
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    west,  east  = sorted(xlim)
    south, north = sorted(ylim)

    if east <= west or north <= south:
        return

    try:
        img, mercator_extent = ctx.bounds2img(
            west, south, east, north,
            source=ctx.providers.OpenStreetMap.Mapnik,
            ll=True,
            zoom="auto",
        )

        transformer = Transformer.from_crs(
            "EPSG:3857", "EPSG:4326", always_xy=True
        )
        west_ll,  south_ll = transformer.transform(mercator_extent[0], mercator_extent[2])
        east_ll,  north_ll = transformer.transform(mercator_extent[1], mercator_extent[3])

        ax.imshow(
            img,
            extent=(west_ll, east_ll, south_ll, north_ll),
            origin="upper",
            alpha=alpha,
            zorder=1,
            interpolation="bilinear",
        )

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
    except Exception as exc:
        print(f"OpenStreetMap basemap skipped: {exc}")


def plot_gridded_map(
    da,
    title,
    cmap="Blues",
    vmin=0,
    vmax=None,
    zero_as_nan=None,
    add_basemap=None,
    city=None,
    figure_size=(10, 5),
    cbar_label=None,
    output_path=None,
    rainfall_alpha=None,
    show=True,
):
    """Plot a lat/lon DataArray with optional OSM basemap and transparent dry pixels."""
    field = prepare_field_for_plot(da, zero_as_nan=zero_as_nan)
    cmap_obj = get_transparent_colormap(cmap)

    fig, ax = plt.subplots(figsize=figure_size)

    ax.set_xlim(float(field.lon.min()), float(field.lon.max()))
    ax.set_ylim(float(field.lat.min()), float(field.lat.max()))

    if add_basemap is None:
        add_basemap = settings.get("plot_use_osm_basemap", True)
    add_openstreetmap_basemap(ax, enabled=add_basemap)

    # Resolve the rainfall-layer alpha. A value below 1 makes the
    # rainfall colours partly transparent so the basemap shows through.
    if rainfall_alpha is None:
        rainfall_alpha = settings.get("plot_rainfall_alpha", 0.7)

    if cbar_label is None:
        cbar_label = field.attrs.get("units", "")

    field.plot.pcolormesh(
        ax=ax,
        x="lon",
        y="lat",
        cmap=cmap_obj,
        vmin=vmin,
        vmax=vmax,
        alpha=rainfall_alpha,
        add_colorbar=True,
        cbar_kwargs={"label": cbar_label},
        zorder=2,
    )

    if city is not None:
        ax.scatter(
            float(city["lon"]),
            float(city["lat"]),
            marker="x",
            s=80,
            label=city["name"],
            zorder=3,
        )
        ax.legend()

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=140, bbox_inches="tight")
        print(f"Saved: {output_path}")

    # When ``show=False`` the caller is expected to add more layers
    # to ``ax`` and call ``plt.show()`` themselves once the figure is
    # complete. Calling ``plt.show()`` here would freeze the figure in
    # Jupyter's inline backend before the additional layers can be drawn.
    if show:
        plt.show()
    return fig, ax


In [ ]:
def get_city_zoom_table(settings):
    """Return city zoom settings as a small table.

    The table is useful in a workshop because it makes all selected cities and
    zoom boxes explicit. Participants can edit the latitude, longitude, or buffer
    and immediately re-run the plots.
    """
    records = []

    for city in settings.get("city_zoom_locations", []):
        buffer_deg = float(settings.get("city_zoom_buffer_degrees", 2.0))
        lat = float(city["lat"])
        lon = float(city["lon"])

        records.append({
            "name": city["name"],
            "country": city.get("country", ""),
            "lat": lat,
            "lon": lon,
            "buffer_degrees": buffer_deg,
            "min_lon": lon - buffer_deg,
            "max_lon": lon + buffer_deg,
            "min_lat": lat - buffer_deg,
            "max_lat": lat + buffer_deg,
            "notes": city.get("notes", ""),
        })

    return pd.DataFrame(records)


def subset_dataarray_around_city(da, city, buffer_degrees):
    """Subset a DataArray to a square lon/lat box around one city."""
    lat = float(city["lat"])
    lon = float(city["lon"])
    buffer_degrees = float(buffer_degrees)

    min_lon = lon - buffer_degrees
    max_lon = lon + buffer_degrees
    min_lat = lat - buffer_degrees
    max_lat = lat + buffer_degrees

    if da.lat.values[0] <= da.lat.values[-1]:
        lat_slice = slice(min_lat, max_lat)
    else:
        lat_slice = slice(max_lat, min_lat)

    return da.sel(lon=slice(min_lon, max_lon), lat=lat_slice)


def plot_city_zoom_field(
    da,
    settings,
    title_prefix,
    file_prefix=None,
    cmap="Blues",
    vmin=0,
    vmax=None,
    cbar_label=None,
    zero_as_nan=None,
):
    """Plot one rainfall (or other) field around each configured city.

    Pass ``vmax`` to keep the colour scale identical to the full-domain plot of
    the same field, so participants can compare zoom and full-domain figures
    directly. If ``vmax`` is None the plot auto-scales to the local maximum.
    Pass ``cmap`` (default 'Blues') to override the colormap, e.g. for
    probability fields ('Blues' or 'YlOrRd').
    """
    if not settings.get("city_zoom_enabled", True):
        print("City zooms are disabled. Set settings['city_zoom_enabled'] = True to enable them.")
        return

    if zero_as_nan is None:
        zero_as_nan = settings.get("plot_zero_as_nan", True)

    buffer_degrees = settings.get("city_zoom_buffer_degrees", 2.0)

    for city in settings.get("city_zoom_locations", []):
        city_da = subset_dataarray_around_city(da, city, buffer_degrees)

        output_path = None
        if file_prefix is not None:
            safe_name = city["name"].lower().replace(" ", "_")
            output_path = figures_folder / f"{file_prefix}_{safe_name}.png"

        plot_gridded_map(
            city_da,
            title=(
                f"{title_prefix}\n"
                f"{city['name']} zoom, +/- {buffer_degrees} degrees"
            ),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            zero_as_nan=zero_as_nan,
            city=city,
            figure_size=(8, 6),
            cbar_label=(
                cbar_label
                if cbar_label is not None
                else city_da.attrs.get("units", "mm h-1")
            ),
            output_path=output_path,
        )


city_zoom_table = get_city_zoom_table(settings)
city_zoom_table


## Load the processed GSMaP observations from Tutorial 1

In [ ]:
# Search for processed GSMaP NetCDF files from Tutorial 1.
processed_files = sorted(processed_folder.glob("GSMaP_*.nc"))
if not processed_files:
    raise FileNotFoundError(
        "No processed GSMaP NetCDF files were found. "
        f"Expected files in {processed_folder.resolve()}. "
        "Re-run Tutorial 1 first."
    )

gsmap_ds = xr.open_mfdataset(processed_files, combine="by_coords").load()
print(f"Loaded {len(processed_files)} processed GSMaP file(s).")
print(gsmap_ds)

## Load the saved nowcast from Tutorial 2

In [ ]:
# Find the most recently saved nowcast NetCDF from Tutorial 2.
nowcast_files = sorted(nowcast_folder.glob("GSMaP_nowcast_*.nc"))
if not nowcast_files:
    raise FileNotFoundError(
        "No nowcast NetCDF was found. "
        f"Expected files in {nowcast_folder.resolve()}. "
        "Re-run Tutorial 2 first to produce a nowcast."
    )

nowcast_file = nowcast_files[-1]
print(f"Using nowcast file: {nowcast_file.name}")

# Filename pattern: GSMaP_nowcast_<method>_<YYYYMMDDTHHMM>.nc
# Extract the nowcast method label so the rest of the notebook can refer to it.
stem_parts = nowcast_file.stem.split("_")
nowcast_method = "_".join(stem_parts[2:-1]) if len(stem_parts) >= 4 else "unknown"
print(f"Nowcast method: {nowcast_method}")

nowcast_ds = xr.open_dataset(nowcast_file).load()
forecast_mean = nowcast_ds["precipitation_rate_nowcast"].mean("member")
max_forecast = forecast_mean.max("time")
print(nowcast_ds)

# Step 16: Calculate ensemble statistics and exceedance probabilities

The nowcast contains multiple ensemble members. These members represent different plausible future rainfall evolutions.

In this step we calculate:

- ensemble mean;
- ensemble maximum;
- ensemble spread;
- probability that rainfall exceeds selected thresholds.

For speed during a workshop, the default spread measure is the ensemble standard deviation. Interquartile spread can be enabled with `calculate_quantiles = True`, but it is slower.

In [ ]:
forecast = nowcast_ds["precipitation_rate_nowcast"]

# Rainfall cannot be negative.
# Negative values can occur as numerical artefacts after nowcast transformations,
# extrapolation or back-transformation. For interpretation and threshold products,
# these values are set to missing values.
forecast = forecast.where(forecast >= 0)

# Load the full nowcast cube once into memory.
# This avoids repeatedly reading/decompressing the same data when calculating
# each ensemble statistic and exceedance probability.
forecast = forecast.load()

ensemble_mean = forecast.mean("member").astype("float32")
ensemble_max = forecast.max("member").astype("float32")
ensemble_std = forecast.std("member").astype("float32")

# Quantiles are useful, but can be slow because xarray has to sort/interpolate
# the ensemble members at every grid cell and every lead time.
# For a workshop demonstration, the ensemble standard deviation is usually fast
# and sufficient to show where members disagree.
calculate_quantiles = False

if calculate_quantiles:
    ensemble_p25 = forecast.quantile(0.25, dim="member").astype("float32")
    ensemble_p75 = forecast.quantile(0.75, dim="member").astype("float32")
    ensemble_spread = (ensemble_p75 - ensemble_p25).astype("float32")
    ensemble_spread.attrs.update({
        "units": "mm h-1",
        "long_name": "Ensemble interquartile spread, p75 minus p25",
        "spread_type": "interquartile_range",
    })
else:
    ensemble_spread = ensemble_std
    ensemble_spread.attrs.update({
        "units": "mm h-1",
        "long_name": "Ensemble standard deviation",
        "spread_type": "standard_deviation",
    })

probability_thresholds_mm_h = [5, 10, 20]
exceedance_probability = xr.Dataset()

for threshold in probability_thresholds_mm_h:
    variable_name = f"prob_exceed_{threshold:g}_mm_h"

    probability = (
        (forecast >= threshold).sum("member") / forecast.sizes["member"] * 100
    ).astype("float32")

    exceedance_probability[variable_name] = probability
    exceedance_probability[variable_name].attrs.update({
        "units": "%",
        "long_name": f"Probability of exceeding {threshold:g} mm h-1",
    })

exceedance_probability

## Pre-compute shared colour scales

The rest of this section will plot the ensemble nowcast in many ways
(probability of exceedance, ensemble spread, max forecast, subplot grid,
animation, city zooms). To make those figures comparable, we compute a
small set of shared colour-scale upper bounds (`vmax`) here, once.

- `intensity_vmax` is used for nowcast intensity plots (mm/h).
- `spread_vmax` is used for ensemble-spread plots (mm/h).
- The probability plots use a fixed scale of 0–100 %.

Each value is the high percentile (default 99) of positive pixels across
the relevant fields, so a single outlier pixel does not wash out the rest
of the figure.


In [ ]:
intensity_vmax = compute_vmax(
    [forecast_mean.isel(time=i) for i in range(forecast_mean.sizes["time"])]
    + [max_forecast],
    override=settings.get("plot_intensity_vmax"),
)
spread_vmax = compute_vmax(
    [ensemble_spread.isel(time=i) for i in range(ensemble_spread.sizes["time"])],
    override=settings.get("plot_spread_vmax"),
)

print(
    f"Shared intensity vmax: {intensity_vmax:.2f} mm/h"
    if intensity_vmax is not None
    else "Shared intensity vmax: no positive forecast pixels found."
)
print(
    f"Shared spread vmax:    {spread_vmax:.2f} mm/h"
    if spread_vmax is not None
    else "Shared spread vmax: no positive spread pixels found."
)


In [ ]:
# Plot exceedance probability for one selected lead time and threshold.
selected_threshold = 10
selected_lead_index = 0
probability_field = exceedance_probability[f"prob_exceed_{selected_threshold:g}_mm_h"].isel(time=selected_lead_index)

plot_gridded_map(
    probability_field,
    title=(
        f"Probability of rainfall >= {selected_threshold:g} mm h-1 | "
        f"lead time {int(probability_field.lead_time.values)} h | "
        f"valid {pd.to_datetime(probability_field.time.values)} UTC"
    ),
    cmap="Blues",
    vmin=0,
    vmax=100,
    # Probability fields can be zero meaningfully, so keep zero visible here.
    zero_as_nan=False,
    cbar_label="%",
)


### Probability zoomed around each city

The same probability-of-exceedance field, but zoomed in around each city.
The probability scale is fixed at 0–100 %, the same as the full-domain
plot above.


In [ ]:
plot_city_zoom_field(
    probability_field,
    settings,
    title_prefix=(
        f"Probability of rainfall >= {selected_threshold:g} mm h-1 | "
        f"lead time {int(probability_field.lead_time.values)} h"
    ),
    file_prefix=f"prob_exceed_{selected_threshold:g}_mm_h_city_zoom",
    cmap="Blues",
    vmin=0,
    vmax=100,
    zero_as_nan=False,
    cbar_label="%",
)


In [ ]:
# Plot ensemble spread for the same lead time. Using ``spread_vmax`` keeps the
# colour scale consistent with the per-city zooms below.
spread_field = ensemble_spread.isel(time=selected_lead_index)
spread_type = spread_field.attrs.get("spread_type", "ensemble_spread")

if spread_type == "interquartile_range":
    spread_title = "Ensemble interquartile spread"
else:
    spread_title = "Ensemble standard deviation"

plot_gridded_map(
    spread_field,
    title=(
        f"{spread_title} | "
        f"lead time {int(spread_field.lead_time.values)} h | "
        f"valid {pd.to_datetime(spread_field.time.values)} UTC"
    ),
    cmap="Blues",
    vmin=0,
    vmax=spread_vmax,
    zero_as_nan=settings.get("plot_zero_as_nan", True),
    cbar_label="mm h-1",
)


### Spread zoomed around each city

The ensemble-spread field, zoomed in around each city. Using
``spread_vmax`` (computed above) means the city zooms share the colour
scale with the full-domain spread plot, so a participant can compare
the zoom and full-domain figures directly.


In [ ]:
plot_city_zoom_field(
    spread_field,
    settings,
    title_prefix=spread_title,
    file_prefix="ensemble_spread_city_zoom",
    cmap="Blues",
    vmin=0,
    vmax=spread_vmax,
    cbar_label="mm h-1",
)


## Exercise 3: Interpret ensemble products

Use the ensemble maps to answer:

1. Where is the forecast rainfall highest?
2. Where is the ensemble spread largest?
3. Where is the probability of exceeding 10 mm h-1 highest?
4. Does a high ensemble mean always imply a high exceedance probability?
5. Which product would be most useful for a warning discussion?

# Step 17: Zoom in around Phnom Penh and Vientiane

The full-domain maps are useful for the regional context. The city zooms are easier to discuss with local stakeholders.

These maps show the maximum forecast rainfall over all lead times around each selected city.

In [ ]:
# Example: zoom in to the maximum nowcast rainfall over all lead times.
# Using ``intensity_vmax`` keeps the colour scale identical to the
# full-domain max-forecast plot from Tutorial 2.
plot_city_zoom_field(
    max_forecast,
    settings,
    title_prefix="Maximum nowcast rainfall rate over all lead times",
    file_prefix="nowcast_max_city_zoom",
    vmax=intensity_vmax,
    cbar_label="mm h-1",
)


### Subplot grid: ensemble mean across all lead times

A single figure showing every forecast lead time of the ensemble mean on
one shared colour scale. This is the same multi-panel layout as in
Tutorial 2, repeated here so participants can verify the saved nowcast
matches what they generated.

All panels share `intensity_vmax`, so colour intensity can be compared
directly across panels.


In [ ]:
n_steps = forecast_mean.sizes["time"]
ncols   = min(3, n_steps)
nrows   = (n_steps + ncols - 1) // ncols

cmap_obj       = get_transparent_colormap("Blues")
rainfall_alpha = settings.get("plot_rainfall_alpha", 0.7)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(5 * ncols, 4 * nrows),
    sharex=True, sharey=True,
)
axes = np.atleast_1d(axes).flatten()

last_mesh = None
for i in range(n_steps):
    ax    = axes[i]
    field = prepare_field_for_plot(
        forecast_mean.isel(time=i),
        zero_as_nan=settings.get("plot_zero_as_nan", True),
    )

    ax.set_xlim(float(field.lon.min()), float(field.lon.max()))
    ax.set_ylim(float(field.lat.min()), float(field.lat.max()))
    add_openstreetmap_basemap(ax)

    last_mesh = ax.pcolormesh(
        field.lon, field.lat, field.values,
        cmap=cmap_obj,
        vmin=0, vmax=intensity_vmax,
        alpha=rainfall_alpha,
        zorder=2,
        shading="auto",
    )

    for city in settings.get("city_zoom_locations", []):
        ax.scatter(
            float(city["lon"]), float(city["lat"]),
            marker="x", s=60, color="black", linewidths=1.5, zorder=3,
        )
        ax.annotate(
            city["name"],
            (float(city["lon"]), float(city["lat"])),
            xytext=(5, 5), textcoords="offset points",
            fontsize=8, zorder=4,
        )

    valid_t = pd.to_datetime(field.time.values)
    lead_h  = int(field.lead_time.values)
    ax.set_title(
        f"+{lead_h} h | {valid_t.strftime('%Y-%m-%d %H:%M')} UTC",
        fontsize=10,
    )

for j in range(n_steps, len(axes)):
    axes[j].axis("off")

fig.subplots_adjust(right=0.90, wspace=0.05, hspace=0.18)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(last_mesh, cax=cbar_ax, label="mm h-1")

fig.supxlabel("Longitude")
fig.supylabel("Latitude")
fig.suptitle(
    "Ensemble-mean nowcast — all lead times (shared colour scale)",
    fontsize=13,
    y=0.995,
)

output_path = figures_folder / "interpret_nowcast_subplots.png"
fig.savefig(output_path, dpi=140, bbox_inches="tight")
print(f"Saved: {output_path}")

plt.show()


### Subplot grid zoomed around each city

A separate subplot grid per city, showing every lead time in the zoom box
around the city. All grids share ``intensity_vmax``, so colour intensity
is comparable across cities and lead times.


In [ ]:
buffer_degrees = settings.get("city_zoom_buffer_degrees", 2.0)

for city in settings.get("city_zoom_locations", []):
    forecast_zoom = subset_dataarray_around_city(forecast_mean, city, buffer_degrees)

    n_steps = forecast_zoom.sizes["time"]
    ncols   = min(3, n_steps)
    nrows   = (n_steps + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4 * ncols, 4 * nrows),
        sharex=True, sharey=True,
    )
    axes = np.atleast_1d(axes).flatten()

    last_mesh = None
    for i in range(n_steps):
        ax    = axes[i]
        field = prepare_field_for_plot(
            forecast_zoom.isel(time=i),
            zero_as_nan=settings.get("plot_zero_as_nan", True),
        )

        ax.set_xlim(float(field.lon.min()), float(field.lon.max()))
        ax.set_ylim(float(field.lat.min()), float(field.lat.max()))
        add_openstreetmap_basemap(ax)

        last_mesh = ax.pcolormesh(
            field.lon, field.lat, field.values,
            cmap=cmap_obj,
            vmin=0, vmax=intensity_vmax,
            alpha=rainfall_alpha,
            zorder=2,
            shading="auto",
        )

        ax.scatter(
            float(city["lon"]), float(city["lat"]),
            marker="x", s=70, color="black", linewidths=1.8, zorder=3,
        )

        valid_t = pd.to_datetime(field.time.values)
        lead_h  = int(field.lead_time.values)
        ax.set_title(
            f"+{lead_h} h | {valid_t.strftime('%Y-%m-%d %H:%M')} UTC",
            fontsize=10,
        )

    for j in range(n_steps, len(axes)):
        axes[j].axis("off")

    fig.subplots_adjust(right=0.90, wspace=0.05, hspace=0.18)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(last_mesh, cax=cbar_ax, label="mm h-1")

    fig.supxlabel("Longitude")
    fig.supylabel("Latitude")
    fig.suptitle(
        f"Ensemble-mean nowcast — "
        f"{city['name']} zoom (±{buffer_degrees}°)",
        fontsize=13,
        y=0.995,
    )

    safe_name = city["name"].lower().replace(" ", "_")
    output_path = figures_folder / f"interpret_nowcast_subplots_{safe_name}.png"
    fig.savefig(output_path, dpi=140, bbox_inches="tight")
    print(f"Saved: {output_path}")

    plt.show()


## Exercise 4: Compare the city zooms

Compare Phnom Penh and Vientiane.

1. Which city has higher forecast rainfall in this example?
2. Is the forecast rainfall close to the city centre or displaced from it?
3. Would the conclusion change if the zoom buffer was smaller or larger?
4. What extra local information would you need before issuing a warning?

# Step 18: Animate the nowcast

Animations are useful for communicating how rainfall moves through time.

The animation below uses `matplotlib.animation.FuncAnimation` rendered
inline with `IPython.display.HTML(ani.to_jshtml())`. This produces a
self-contained HTML/JavaScript player that works in JupyterLab, classic
Jupyter, and VS Code, without requiring `ffmpeg`.

Three details that matter for the animation to render correctly in
Jupyter:

- We close the figure with `plt.close(fig)` immediately after creation.
  Otherwise Jupyter would display the static initial figure AND the
  animation below it.
- We use `blit=False` because we update both the rainfall layer and the
  title each frame.
- The basemap is drawn once before the loop. Only the rainfall layer is
  updated per frame, so OSM tiles are not re-fetched.


In [ ]:
forecast_to_animate = forecast_mean   # 3D: (time, lat, lon)
n_frames            = forecast_to_animate.sizes["time"]
animate_vmax        = intensity_vmax  # share the colour scale

fig, ax = plt.subplots(figsize=(10, 5))
plt.close(fig)   # critical: prevents the static initial figure from also
                 # being shown above the animation in Jupyter.

ax.set_xlim(
    float(forecast_to_animate.lon.min()),
    float(forecast_to_animate.lon.max()),
)
ax.set_ylim(
    float(forecast_to_animate.lat.min()),
    float(forecast_to_animate.lat.max()),
)

add_openstreetmap_basemap(ax)

cmap_obj       = get_transparent_colormap("Blues")
rainfall_alpha = settings.get("plot_rainfall_alpha", 0.7)

field0 = prepare_field_for_plot(
    forecast_to_animate.isel(time=0),
    zero_as_nan=settings.get("plot_zero_as_nan", True),
)
mesh = ax.pcolormesh(
    field0.lon, field0.lat, field0.values,
    cmap=cmap_obj,
    vmin=0, vmax=animate_vmax,
    alpha=rainfall_alpha,
    zorder=2,
    shading="auto",
)

fig.colorbar(mesh, ax=ax, label="mm h-1")

for city in settings.get("city_zoom_locations", []):
    ax.scatter(
        float(city["lon"]), float(city["lat"]),
        marker="x", s=60, color="black", linewidths=1.5, zorder=3,
    )
    ax.annotate(
        city["name"],
        (float(city["lon"]), float(city["lat"])),
        xytext=(5, 5), textcoords="offset points",
        fontsize=9, zorder=4,
    )

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
title = ax.set_title("")


def update(i):
    field = prepare_field_for_plot(
        forecast_to_animate.isel(time=i),
        zero_as_nan=settings.get("plot_zero_as_nan", True),
    )
    mesh.set_array(field.values.ravel())
    valid_t = pd.to_datetime(field.time.values)
    lead_h  = int(field.lead_time.values)
    title.set_text(
        f"Ensemble-mean nowcast | +{lead_h} h | "
        f"valid {valid_t.strftime('%Y-%m-%d %H:%M')} UTC"
    )
    return (mesh, title)


ani = FuncAnimation(
    fig, update,
    frames=n_frames,
    interval=800,
    blit=False,
)

# --- Optional: save as a GIF (requires `imageio` + `pillow`, both already imported). ---
# gif_path = figures_folder / "gsmap_nowcast_animation.gif"
# ani.save(gif_path, writer="pillow", fps=2)
# print(f"Saved animation: {gif_path}")

HTML(ani.to_jshtml())


### Animations zoomed around each city

One animation per city, showing the forecast evolution over the city's
zoom box. As before, the basemap is drawn once per figure and only the
rainfall layer is updated each frame.

`display(HTML(...))` is used inside the loop so that **multiple**
animations render in the same cell. Without `display(...)`, only the
last animation would be shown (Jupyter only auto-renders the cell's
final expression).


In [ ]:
buffer_degrees = settings.get("city_zoom_buffer_degrees", 2.0)
cmap_obj       = get_transparent_colormap("Blues")
rainfall_alpha = settings.get("plot_rainfall_alpha", 0.7)


def _build_city_animation(forecast_zoom, city, intensity_vmax):
    """Build one FuncAnimation for the given city's zoomed forecast."""
    n_frames = forecast_zoom.sizes["time"]

    fig, ax = plt.subplots(figsize=(7, 6))
    plt.close(fig)   # prevents the static initial figure from also
                     # appearing above the animation in Jupyter.

    ax.set_xlim(float(forecast_zoom.lon.min()), float(forecast_zoom.lon.max()))
    ax.set_ylim(float(forecast_zoom.lat.min()), float(forecast_zoom.lat.max()))

    add_openstreetmap_basemap(ax)

    field0 = prepare_field_for_plot(
        forecast_zoom.isel(time=0),
        zero_as_nan=settings.get("plot_zero_as_nan", True),
    )
    mesh = ax.pcolormesh(
        field0.lon, field0.lat, field0.values,
        cmap=cmap_obj,
        vmin=0, vmax=intensity_vmax,
        alpha=rainfall_alpha,
        zorder=2,
        shading="auto",
    )

    fig.colorbar(mesh, ax=ax, label="mm h-1")

    ax.scatter(
        float(city["lon"]), float(city["lat"]),
        marker="x", s=80, color="black", linewidths=2, zorder=3,
    )
    ax.annotate(
        city["name"],
        (float(city["lon"]), float(city["lat"])),
        xytext=(7, 7), textcoords="offset points",
        fontsize=10, zorder=4,
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    title = ax.set_title("")

    # Factory pattern: bind ``forecast_zoom``, ``mesh``, ``title``, ``city`` by
    # argument so each city's animation gets its own closure state. Defining
    # ``update`` inline in the for-loop would create closures that all share
    # the LAST iteration's variables.
    def update(i):
        field = prepare_field_for_plot(
            forecast_zoom.isel(time=i),
            zero_as_nan=settings.get("plot_zero_as_nan", True),
        )
        mesh.set_array(field.values.ravel())
        valid_t = pd.to_datetime(field.time.values)
        lead_h  = int(field.lead_time.values)
        title.set_text(
            f"{city['name']} | +{lead_h} h | "
            f"valid {valid_t.strftime('%Y-%m-%d %H:%M')} UTC"
        )
        return (mesh, title)

    return FuncAnimation(
        fig, update, frames=n_frames, interval=800, blit=False,
    )


for city in settings.get("city_zoom_locations", []):
    forecast_zoom = subset_dataarray_around_city(forecast_mean, city, buffer_degrees)

    ani = _build_city_animation(forecast_zoom, city, intensity_vmax)

    display(HTML(
        f"<h4 style='margin-top:1em'>"
        f"{city['name']} ({city.get('country', '')}) "
        f"— zoom animation"
        f"</h4>"
    ))
    display(HTML(ani.to_jshtml()))


# Step 19: Extract nowcast time series for point locations

For flood forecasting and warning discussions, point time series are often easier to interpret than maps.

Here we extract the nearest grid cell for the selected city locations. In real applications, you may want to use verified station coordinates, drainage hotspots, or catchment outlet points instead.

In [ ]:
# Use the same city locations as point-extraction examples.
# For real applications, replace these with verified station coordinates,
# drainage hotspots, or catchment outlet points.
points = pd.DataFrame(
    [
        {
            "name": city["name"],
            "country": city.get("country", ""),
            "lat": city["lat"],
            "lon": city["lon"],
        }
        for city in settings.get("city_zoom_locations", [])
    ]
)

points


In [ ]:
def extract_point_nowcast(nowcast_ds, points):
    """Extract nearest-cell nowcast values for a list of point locations."""
    records = []
    forecast_mean = nowcast_ds["precipitation_rate_nowcast"].mean("member")

    for _, point in points.iterrows():
        series = forecast_mean.sel(lat=point["lat"], lon=point["lon"], method="nearest")

        for i in range(series.sizes["time"]):
            value = series.isel(time=i)
            records.append({
                "name": point["name"],
                "input_lat": point["lat"],
                "input_lon": point["lon"],
                "grid_lat": float(value.lat.values),
                "grid_lon": float(value.lon.values),
                "lead_time_h": int(value.lead_time.values),
                "valid_time_utc": pd.to_datetime(value.time.values),
                "precipitation_rate_mm_h": float(value.values),
            })

    return pd.DataFrame(records)


point_nowcast = extract_point_nowcast(nowcast_ds, points)
point_csv = nowcast_folder / "gsmap_point_nowcast.csv"
point_nowcast.to_csv(point_csv, index=False)
print(f"Saved point nowcast table: {point_csv}")
point_nowcast.head(12)


# Step 20: Create threshold-probability tables for the cities

A threshold-probability table translates the ensemble nowcast into a compact warning-oriented format.

For each city and lead time, the table shows the probability of exceeding selected rainfall thresholds.

In [ ]:
def extract_city_threshold_probabilities(nowcast_ds, points, thresholds_mm_h):
    """Extract ensemble threshold probabilities at nearest grid cells for selected points."""
    forecast = nowcast_ds["precipitation_rate_nowcast"]
    records = []

    for _, point in points.iterrows():
        point_forecast = forecast.sel(
            lat=point["lat"],
            lon=point["lon"],
            method="nearest",
        )

        for time_index in range(point_forecast.sizes["time"]):
            lead_slice = point_forecast.isel(time=time_index)

            record = {
                "name": point["name"],
                "country": point.get("country", ""),
                "input_lat": point["lat"],
                "input_lon": point["lon"],
                "grid_lat": float(lead_slice.lat.values),
                "grid_lon": float(lead_slice.lon.values),
                "lead_time_h": int(lead_slice.lead_time.values),
                "valid_time_utc": pd.to_datetime(lead_slice.time.values),
                "ensemble_mean_mm_h": float(lead_slice.mean("member").values),
                "ensemble_max_mm_h": float(lead_slice.max("member").values),
            }

            for threshold in thresholds_mm_h:
                column = f"prob_exceed_{threshold:g}_mm_h_percent"
                record[column] = float((lead_slice >= threshold).mean("member").values * 100)

            records.append(record)

    return pd.DataFrame(records)


city_threshold_probabilities = extract_city_threshold_probabilities(
    nowcast_ds,
    points,
    thresholds_mm_h=[5, 10, 20, 30],
)

city_threshold_csv = nowcast_folder / "gsmap_city_threshold_probabilities.csv"
city_threshold_probabilities.to_csv(city_threshold_csv, index=False)
print(f"Saved city threshold-probability table: {city_threshold_csv}")

city_threshold_probabilities


# Step 21: Plot ensemble time series for the cities

The ensemble time-series plot shows each ensemble member, the ensemble mean, and a 10th to 90th percentile envelope.

This plot is useful for explaining forecast uncertainty to participants. A narrow envelope means the members agree. A wide envelope means the forecast is more uncertain.

In [ ]:
def plot_ensemble_timeseries_for_points(nowcast_ds, points, output_folder=None):
    """Plot ensemble rainfall nowcast time series for each selected point.

    The function uses nearest-neighbour extraction from the gridded nowcast.
    For real validation studies, consider averaging over a small buffer around
    the station or city instead of using one grid cell only.
    """
    forecast = nowcast_ds["precipitation_rate_nowcast"]

    for _, point in points.iterrows():
        point_forecast = forecast.sel(
            lat=point["lat"],
            lon=point["lon"],
            method="nearest",
        )

        times = pd.to_datetime(point_forecast["time"].values)
        members = point_forecast.values  # shape: member, time

        ensemble_mean = point_forecast.mean("member").values
        ensemble_p10 = point_forecast.quantile(0.10, dim="member").values
        ensemble_p90 = point_forecast.quantile(0.90, dim="member").values

        fig, ax = plt.subplots(figsize=(11, 5))

        for member_index in range(point_forecast.sizes["member"]):
            ax.plot(
                times,
                members[member_index, :],
                linewidth=0.8,
                alpha=0.45,
            )

        ax.fill_between(
            times,
            ensemble_p10,
            ensemble_p90,
            alpha=0.25,
            label="10th-90th percentile range",
        )
        ax.plot(
            times,
            ensemble_mean,
            linewidth=2.5,
            label="Ensemble mean",
        )

        ax.set_title(
            f"Ensemble nowcast time series: {point['name']}\n"
            f"nearest grid cell: {float(point_forecast.lat.values):.3f}, "
            f"{float(point_forecast.lon.values):.3f}"
        )
        ax.set_xlabel("Forecast valid time (UTC)")
        ax.set_ylabel("Rainfall rate (mm h-1)")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.autofmt_xdate()
        plt.show()

        if output_folder is not None:
            output_path = output_folder / f"ensemble_timeseries_{point['name'].replace(' ', '_')}.png"
            fig.savefig(output_path, dpi=150, bbox_inches="tight")
            print(f"Saved ensemble time-series plot: {output_path}")


plot_ensemble_timeseries_for_points(nowcast_ds, points, output_folder=figures_folder)

## Exercise 5: Interpret the ensemble time-series plots

For each city, answer:

1. At what lead time is the ensemble mean highest?
2. Do the ensemble members agree, or do they diverge strongly?
3. Is the uncertainty larger at short lead times or longer lead times?
4. Would you use the ensemble mean, the maximum member, or the exceedance probability for warning communication?

# Step 22: Compare GSMaP and the nowcast against a rain-gauge CSV

In practice, satellite rainfall and nowcasts should be compared against independent observations where possible.

This section demonstrates the workflow using a simple rain-gauge CSV. If no file is available, the notebook creates a small synthetic example CSV so the code can still be demonstrated during the workshop.

For a real case study, replace the example file with your own CSV using the following columns:

```text
station_id, station_name, latitude, longitude, time_utc, rainfall_mm
```

The `time_utc` column should use UTC timestamps. The `rainfall_mm` column should represent the observed rainfall amount for the corresponding hour.

In [ ]:
example_gauge_csv = data_folder / "example_rain_gauge_observations.csv"

if not example_gauge_csv.exists():
    # Create a synthetic example based on the available GSMaP time axis.
    # This is only for demonstrating the workflow. It should not be used as real validation data.
    observed_times = pd.to_datetime(gsmap_ds["time"].values)
    example_records = []

    for _, point in points.iterrows():
        nearest_gsmap = gsmap_ds["precipitation_rate"].sel(
            lat=point["lat"],
            lon=point["lon"],
            method="nearest",
        )

        for time_index, time_value in enumerate(observed_times):
            gsmap_value = float(nearest_gsmap.isel(time=time_index).values)

            # Add a small deterministic perturbation so the gauge is not identical to GSMaP.
            # The sine term keeps the example reproducible without using random numbers.
            synthetic_gauge_value = max(0.0, gsmap_value * 1.10 + 0.7 * np.sin(time_index))

            example_records.append({
                "station_id": point["name"].lower().replace(" ", "_"),
                "station_name": f"Example gauge near {point['name']}",
                "latitude": point["lat"],
                "longitude": point["lon"],
                "time_utc": time_value.strftime("%Y-%m-%d %H:%M:%S"),
                "rainfall_mm": round(synthetic_gauge_value, 3),
            })

    pd.DataFrame(example_records).to_csv(example_gauge_csv, index=False)
    print(f"Created example gauge CSV: {example_gauge_csv}")
else:
    print(f"Using existing gauge CSV: {example_gauge_csv}")

rain_gauge_df = pd.read_csv(example_gauge_csv)
rain_gauge_df["time_utc"] = pd.to_datetime(rain_gauge_df["time_utc"], utc=True).dt.tz_convert(None)

rain_gauge_df.head()

## Step 22.1: Extract collocated GSMaP observations and nowcast values

For each gauge record, we extract the nearest GSMaP grid cell.

For gauge records that overlap with the nowcast valid times, we also extract the nowcast ensemble statistics at the nearest grid cell.

In [ ]:
def extract_gsmap_at_gauge_locations(gsmap_ds, rain_gauge_df):
    """Extract nearest GSMaP observed rainfall values for each gauge record."""
    records = []

    for _, row in rain_gauge_df.iterrows():
        gsmap_point = gsmap_ds["precipitation_rate"].sel(
            time=row["time_utc"],
            lat=row["latitude"],
            lon=row["longitude"],
            method="nearest",
        )

        records.append({
            "station_id": row["station_id"],
            "station_name": row["station_name"],
            "time_utc": row["time_utc"],
            "gauge_rainfall_mm": row["rainfall_mm"],
            "gsmap_rainfall_mm_h": float(gsmap_point.values),
            "gsmap_grid_lat": float(gsmap_point.lat.values),
            "gsmap_grid_lon": float(gsmap_point.lon.values),
        })

    return pd.DataFrame(records)


def extract_nowcast_at_gauge_locations(nowcast_ds, rain_gauge_df):
    """Extract nearest nowcast ensemble statistics for each gauge/time record.

    Only gauge records with times that can be matched to the nowcast valid-time
    axis are returned. Times outside the nowcast period are skipped.
    """
    forecast = nowcast_ds["precipitation_rate_nowcast"]
    nowcast_times = pd.to_datetime(nowcast_ds["time"].values)
    records = []

    for _, row in rain_gauge_df.iterrows():
        gauge_time = pd.to_datetime(row["time_utc"])

        if gauge_time not in set(nowcast_times):
            continue

        point_forecast = forecast.sel(
            time=gauge_time,
            lat=row["latitude"],
            lon=row["longitude"],
            method="nearest",
        )

        records.append({
            "station_id": row["station_id"],
            "time_utc": gauge_time,
            "nowcast_ensemble_mean_mm_h": float(point_forecast.mean("member").values),
            "nowcast_ensemble_p10_mm_h": float(point_forecast.quantile(0.10, dim="member").values),
            "nowcast_ensemble_p90_mm_h": float(point_forecast.quantile(0.90, dim="member").values),
            "nowcast_grid_lat": float(point_forecast.lat.values),
            "nowcast_grid_lon": float(point_forecast.lon.values),
        })

    return pd.DataFrame(records)


gauge_gsmap_comparison = extract_gsmap_at_gauge_locations(gsmap_ds, rain_gauge_df)
gauge_nowcast_comparison = extract_nowcast_at_gauge_locations(nowcast_ds, rain_gauge_df)

if len(gauge_nowcast_comparison) > 0:
    gauge_comparison = gauge_gsmap_comparison.merge(
        gauge_nowcast_comparison,
        on=["station_id", "time_utc"],
        how="left",
    )
else:
    gauge_comparison = gauge_gsmap_comparison.copy()
    print("No rain-gauge times overlap with the nowcast valid-time axis.")

comparison_csv = nowcast_folder / "rain_gauge_gsmap_nowcast_comparison.csv"
gauge_comparison.to_csv(comparison_csv, index=False)
print(f"Saved gauge comparison table: {comparison_csv}")

gauge_comparison.head(12)

## Step 22.2: Calculate simple validation scores

The scores below are deliberately simple. They are intended for training and first inspection, not for a full scientific validation study.

We calculate:

- bias;
- mean absolute error;
- root mean square error;
- correlation.

In [ ]:
def calculate_simple_scores(df, observation_column, estimate_column):
    """Calculate simple continuous validation scores."""
    valid = df[[observation_column, estimate_column]].dropna()

    if len(valid) == 0:
        return {
            "n": 0,
            "bias_mm": np.nan,
            "mae_mm": np.nan,
            "rmse_mm": np.nan,
            "correlation": np.nan,
        }

    error = valid[estimate_column] - valid[observation_column]

    return {
        "n": int(len(valid)),
        "bias_mm": float(error.mean()),
        "mae_mm": float(error.abs().mean()),
        "rmse_mm": float(np.sqrt((error ** 2).mean())),
        "correlation": float(valid[observation_column].corr(valid[estimate_column])) if len(valid) > 1 else np.nan,
    }


score_records = []

for station_id, station_df in gauge_comparison.groupby("station_id"):
    station_name = station_df["station_name"].iloc[0]

    gsmap_scores = calculate_simple_scores(
        station_df,
        observation_column="gauge_rainfall_mm",
        estimate_column="gsmap_rainfall_mm_h",
    )
    gsmap_scores.update({
        "station_id": station_id,
        "station_name": station_name,
        "product": "GSMaP observed",
    })
    score_records.append(gsmap_scores)

    if "nowcast_ensemble_mean_mm_h" in station_df.columns:
        nowcast_scores = calculate_simple_scores(
            station_df,
            observation_column="gauge_rainfall_mm",
            estimate_column="nowcast_ensemble_mean_mm_h",
        )
        nowcast_scores.update({
            "station_id": station_id,
            "station_name": station_name,
            "product": "Nowcast ensemble mean",
        })
        score_records.append(nowcast_scores)

validation_scores = pd.DataFrame(score_records)[
    ["station_id", "station_name", "product", "n", "bias_mm", "mae_mm", "rmse_mm", "correlation"]
]

validation_scores

## Step 22.3: Plot gauge, GSMaP and nowcast time series

These plots show how the gridded product and nowcast compare against the gauge time series.

Remember that this is a nearest-grid-cell comparison. For real validation, consider station representativeness, timing, gauge quality control, and spatial averaging.

In [ ]:
def plot_gauge_gsmap_nowcast_comparison(gauge_comparison):
    """Plot gauge observations against GSMaP and nowcast values for each station."""
    for station_id, station_df in gauge_comparison.groupby("station_id"):
        station_df = station_df.sort_values("time_utc")
        station_name = station_df["station_name"].iloc[0]

        fig, ax = plt.subplots(figsize=(11, 5))

        ax.plot(
            station_df["time_utc"],
            station_df["gauge_rainfall_mm"],
            marker="o",
            linewidth=2.0,
            label="Rain gauge",
        )
        ax.plot(
            station_df["time_utc"],
            station_df["gsmap_rainfall_mm_h"],
            marker="s",
            linewidth=2.0,
            label="GSMaP nearest grid cell",
        )

        if "nowcast_ensemble_mean_mm_h" in station_df.columns:
            nowcast_valid = station_df.dropna(subset=["nowcast_ensemble_mean_mm_h"])

            if len(nowcast_valid) > 0:
                ax.fill_between(
                    nowcast_valid["time_utc"],
                    nowcast_valid["nowcast_ensemble_p10_mm_h"],
                    nowcast_valid["nowcast_ensemble_p90_mm_h"],
                    alpha=0.25,
                    label="Nowcast 10th-90th percentile",
                )
                ax.plot(
                    nowcast_valid["time_utc"],
                    nowcast_valid["nowcast_ensemble_mean_mm_h"],
                    marker="^",
                    linewidth=2.0,
                    label="Nowcast ensemble mean",
                )

        ax.set_title(f"Rain-gauge comparison: {station_name}")
        ax.set_xlabel("Time (UTC)")
        ax.set_ylabel("Hourly rainfall / rainfall rate (mm)")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.autofmt_xdate()
        plt.show()

        output_path = figures_folder / f"gauge_comparison_{station_id}.png"
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"Saved gauge comparison plot: {output_path}")


plot_gauge_gsmap_nowcast_comparison(gauge_comparison)

## Exercise 6: Compare the gridded rainfall product with gauges

Use the gauge-comparison table and plots to discuss:

1. Does GSMaP overestimate or underestimate the gauge rainfall?
2. Does the nowcast improve or degrade the comparison at overlapping times?
3. Are the timing differences more important than the rainfall-intensity differences?
4. Would one gauge be enough to validate a city-scale nowcast?

# Step 23: Select simple alert thresholds

This table checks whether the maximum nowcast rainfall at each point exceeds selected thresholds.

This is a simplified warning exercise. Real alert thresholds should be calibrated with local hydrology, impacts, drainage capacity, vulnerability, and historical events.

In [ ]:
thresholds_mm_h = [1, 5, 10, 20, 30]

threshold_records = []
for threshold in thresholds_mm_h:
    exceedance = point_nowcast.groupby("name")["precipitation_rate_mm_h"].max().reset_index()
    exceedance["threshold_mm_h"] = threshold
    exceedance["exceeds_threshold"] = exceedance["precipitation_rate_mm_h"] >= threshold
    threshold_records.append(exceedance)

threshold_table = pd.concat(threshold_records, ignore_index=True)
threshold_table = threshold_table.rename(columns={"precipitation_rate_mm_h": "max_nowcast_precipitation_mm_h"})
threshold_table


# Step 24: Optional validation against withheld GSMaP observations

If the nowcast reference time is earlier than the last downloaded GSMaP timestep, some later GSMaP fields can be used as a simple validation target.

This is not independent validation, because the nowcast and validation data come from the same satellite product. Still, it is useful for teaching how forecast verification works.

In [ ]:
def validate_nowcast_against_observed(nowcast_ds, observed_ds, threshold_mm_h=5.0):
    """Compare ensemble-mean nowcast rainfall with observed GSMaP at matching times."""
    forecast_mean = nowcast_ds["precipitation_rate_nowcast"].mean("member")

    forecast_times = pd.to_datetime(forecast_mean.time.values)
    observed_times = pd.to_datetime(observed_ds.time.values)
    common_times = np.intersect1d(forecast_times.values.astype("datetime64[ns]"), observed_times.values.astype("datetime64[ns]"))

    if len(common_times) == 0:
        print("No overlapping forecast valid times and observed GSMaP times were found.")
        print("For validation, set settings['nowcast_reference_time'] earlier than the latest downloaded timestep.")
        return pd.DataFrame(), None

    records = []
    error_fields = []

    for valid_time in common_times:
        forecast_field = forecast_mean.sel(time=valid_time)
        observed_field = observed_ds["precipitation_rate"].sel(time=valid_time)

        # Align coordinates exactly before calculating errors.
        forecast_field, observed_field = xr.align(forecast_field, observed_field, join="inner")

        error = forecast_field - observed_field
        error_fields.append(error.assign_coords(time=valid_time).expand_dims("time"))

        forecast_event = forecast_field >= threshold_mm_h
        observed_event = observed_field >= threshold_mm_h

        hits = int((forecast_event & observed_event).sum().values)
        misses = int((~forecast_event & observed_event).sum().values)
        false_alarms = int((forecast_event & ~observed_event).sum().values)
        correct_negatives = int((~forecast_event & ~observed_event).sum().values)

        records.append({
            "valid_time_utc": pd.to_datetime(valid_time),
            "lead_time_h": int(forecast_field.lead_time.values),
            "mean_observed_mm_h": float(observed_field.mean(skipna=True).values),
            "mean_forecast_mm_h": float(forecast_field.mean(skipna=True).values),
            "bias_mm_h": float(error.mean(skipna=True).values),
            "mae_mm_h": float(abs(error).mean(skipna=True).values),
            "max_observed_mm_h": float(observed_field.max(skipna=True).values),
            "max_forecast_mm_h": float(forecast_field.max(skipna=True).values),
            "threshold_mm_h": threshold_mm_h,
            "hits": hits,
            "misses": misses,
            "false_alarms": false_alarms,
            "correct_negatives": correct_negatives,
        })

    validation_table = pd.DataFrame(records)
    error_ds = xr.concat(error_fields, dim="time").to_dataset(name="forecast_error_mm_h")

    return validation_table, error_ds


validation_table, validation_error_ds = validate_nowcast_against_observed(
    nowcast_ds,
    gsmap_ds,
    threshold_mm_h=5.0,
)

if not validation_table.empty:
    validation_csv = nowcast_folder / "gsmap_nowcast_validation.csv"
    validation_table.to_csv(validation_csv, index=False)
    print(f"Saved validation table: {validation_csv}")

validation_table


In [ ]:
# Plot forecast error for the first available validation time.
if validation_error_ds is not None:
    error_field = validation_error_ds["forecast_error_mm_h"].isel(time=0)

    plot_gridded_map(
        error_field,
        title=(
            "Forecast error: ensemble mean nowcast minus observed GSMaP | "
            f"valid {pd.to_datetime(error_field.time.values)} UTC"
        ),
        cmap="RdBu_r",
        # Error fields contain meaningful positive and negative values.
        # Do not convert zeros to NaN here.
        zero_as_nan=False,
        cbar_label="mm h-1",
    )


## Exercise 7: Validate the nowcast

Set `settings["nowcast_reference_time"]` to an earlier time and re-run the nowcast.

Then answer:

1. How many validation times are available?
2. Does forecast error increase with lead time?
3. Are misses or false alarms more common for the selected threshold?
4. How would the validation change if rain-gauge data were available?

# Step 25: Export additional products to GeoTIFF

GeoTIFF output is useful when the nowcast needs to be inspected in GIS software or shared with colleagues who do not use Python.

This step exports the ensemble mean and exceedance-probability maps for each lead time.

In [ ]:
def export_dataarray_time_slices_to_geotiff(da, output_folder, filename_prefix):
    """Export each time slice of a lat/lon DataArray to a separate GeoTIFF."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    exported_files = []

    for time_index in range(da.sizes["time"]):
        field = da.isel(time=time_index)

        if "lead_time" in field.coords:
            lead_time_h = int(field.lead_time.values)
            lead_text = f"lead_{lead_time_h:02d}h"
        else:
            lead_text = f"time_{time_index:02d}"

        valid_time = pd.to_datetime(field.time.values).strftime("%Y%m%dT%H%M")
        out_tif = output_folder / f"{filename_prefix}_{lead_text}_{valid_time}.tif"

        # rioxarray expects explicit spatial dimension names and CRS.
        field_to_write = (
            field
            .rio.write_crs("EPSG:4326")
            .rio.set_spatial_dims(x_dim="lon", y_dim="lat")
        )

        field_to_write.rio.to_raster(out_tif)
        exported_files.append(out_tif)

    return exported_files


geotiff_output_folder = nowcast_folder / "geotiff_exports"

mean_geotiffs = export_dataarray_time_slices_to_geotiff(
    ensemble_mean,
    geotiff_output_folder,
    "gsmap_nowcast_ensemble_mean",
)

probability_geotiffs = []
for threshold in probability_thresholds_mm_h:
    probability_geotiffs.extend(
        export_dataarray_time_slices_to_geotiff(
            exceedance_probability[f"prob_exceed_{threshold:g}_mm_h"],
            geotiff_output_folder,
            f"gsmap_nowcast_probability_exceed_{threshold:g}_mm_h",
        )
    )

print(f"Exported {len(mean_geotiffs)} ensemble-mean GeoTIFF files.")
print(f"Exported {len(probability_geotiffs)} probability GeoTIFF files.")
print(f"Output folder: {geotiff_output_folder}")


## Exercise 8: Export and reuse the results

Open one exported GeoTIFF in QGIS or another GIS tool.

Check:

1. Is the spatial location correct?
2. Are the values plausible?
3. Which product is easiest to interpret: rainfall rate, exceedance probability, or point time series?
4. What format would be most useful for a flood model or warning dashboard?

# End of Tutorial 3

You have now exercised the full GSMaP nowcasting workflow:

1. **Tutorial 1** — downloaded and preprocessed GSMaP rainfall.
2. **Tutorial 2** — produced an ensemble pySTEPS nowcast and saved it as NetCDF.
3. **Tutorial 3** *(this notebook)* — computed ensemble products, extracted city-level time series and threshold probabilities, compared against rain gauges, validated against withheld observations, and exported operational products to GeoTIFF.

From here you can:

- Re-run all three tutorials with a different event by changing the `event_start_utc` and `event_end_utc` settings in Tutorial 1.
- Replace the example rain-gauge CSV with real station observations from your national meteorological service.
- Use the saved NetCDF and GeoTIFF outputs as input to a downstream urban flash-flood model for Vientiane or Phnom Penh.